In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from rdkit.Chem import Descriptors
from rdkit.Chem import AllChem
from rdkit import Chem

In [ ]:
modeling_data = pd.read_excel("../../dataset/modeling data.xlsx")
plant_data = pd.read_excel("../../dataset/Plant_taxonomy_trait.xlsx")

In [4]:
modeling_data.shape

(1388, 18)

In [ ]:
# ===============================
# Merge plant taxonomy + traits by species
# ===============================
df = modeling_data.merge(
    plant_data,
    on="Species",
    how="left"
)
print(df.shape)
print(df.head())

(1388, 31)
   Reference                      Sample                 Soil source  \
0          1  Industrially Impacted Soil  industrially Impacted Soil   
1          1  Industrially Impacted Soil  industrially Impacted Soil   
2          1  Industrially Impacted Soil  industrially Impacted Soil   
3          1  Industrially Impacted Soil  industrially Impacted Soil   
4          1  Industrially Impacted Soil  industrially Impacted Soil   

           ES   PFAS PFAS class   PH   SOC   CEC  Sand  ...      Order  \
0  Greenhouse   PFBA       PFCA  6.4  2.24  16.1  50.0  ...  Asterales   
1  Greenhouse   PFBA       PFCA  6.4  2.24  16.1  50.0  ...  Solanales   
2  Greenhouse  PFPeA       PFCA  6.4  2.24  16.1  50.0  ...  Asterales   
3  Greenhouse  PFPeA       PFCA  6.4  2.24  16.1  50.0  ...  Solanales   
4  Greenhouse  PFHxA       PFCA  6.4  2.24  16.1  50.0  ...  Asterales   

   monocot woody herb crop  vegetable  legume  grass perennial edible  
0      0.0   0.0  1.0  1.0        1.0  

In [5]:
df["group"] = df["PFAS"].astype(str) + "__" + df["Species"].astype(str)
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(df, groups=df["group"])
)

train_df = df.iloc[train_idx].reset_index(drop=True)
test_df  = df.iloc[test_idx].reset_index(drop=True)

In [ ]:
train_groups = set(train_df["group"])
test_groups  = set(test_df["group"])

overlap = train_groups & test_groups
print("combinations with repetition:", len(overlap))   # 应为0

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

print("Train groups:", len(train_groups))
print("Test groups :", len(test_groups))

重复组合数: 0
Train shape: (1129, 32)
Test shape : (259, 32)
Train groups: 100
Test groups : 25


In [ ]:
pfas = pd.read_excel("../../dataset/PFAS.xlsx")
descriptor_names = [d[0] for d in Descriptors._descList]

rows = []

for smi in pfas["Raw_SMILES"]:
    mol = Chem.MolFromSmiles(smi)

    if mol is None:
        rows.append([np.nan] * len(descriptor_names))
        continue

    vals = []
    for name in descriptor_names:
        func = Descriptors.__dict__[name]
        try:
            vals.append(func(mol))
        except:
            vals.append(np.nan)

    rows.append(vals)

rdkit_df = pd.DataFrame(rows, columns=descriptor_names)
rdkit_df["PFAS"] = pfas["PFAS_name"]

#rdkit_df.to_excel("../../dataset/rdkit_descriptors.xlsx", index=False)

In [10]:
train = train_df.merge(rdkit_df, on="PFAS", how="left")

In [ ]:
rd_cols = descriptor_names
X_rd = train[rd_cols].copy()
from sklearn.feature_selection import VarianceThreshold

vt = VarianceThreshold(threshold=0)
X_rd = pd.DataFrame(
    vt.fit_transform(X_rd),
    columns=X_rd.columns[vt.get_support()]
)
corr = X_rd.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

drop_cols = [c for c in upper.columns if any(upper[c] > 0.95)]
X_rd = X_rd.drop(columns=drop_cols)
selected_rd_cols = X_rd.columns.tolist()
print("Number of features to keep:", len(selected_rd_cols))
selected_rd_cols

保留特征数: 23


['MaxAbsEStateIndex',
 'MinAbsEStateIndex',
 'qed',
 'SPS',
 'MaxPartialCharge',
 'MinPartialCharge',
 'MaxAbsPartialCharge',
 'MinAbsPartialCharge',
 'BCUT2D_MWLOW',
 'BCUT2D_MRLOW',
 'HallKierAlpha',
 'Ipc',
 'Kappa3',
 'PEOE_VSA13',
 'PEOE_VSA3',
 'PEOE_VSA8',
 'SMR_VSA4',
 'SlogP_VSA12',
 'EState_VSA8',
 'EState_VSA9',
 'VSA_EState7',
 'FractionCSP3',
 'NHOHCount']

In [ ]:
train_base = train.drop(columns=rd_cols, errors="ignore")
train_final = pd.concat(
    [train_base.reset_index(drop=True),
     X_rd.reset_index(drop=True)],
    axis=1
)
train_final.to_excel(
    "../dataset/train set.xlsx",
    index=False
)

print("Saved: ../../dataset/train set.xlsx")
print("Final shape:", train_final.shape)

Saved: ../dataset/train set.xlsx
Final shape: (1129, 55)


In [ ]:
test = test_df.merge(rdkit_df, on="PFAS", how="left")
X_test_rd = test[selected_rd_cols].copy()
test_base = test.drop(columns=rd_cols, errors="ignore")
test_final = pd.concat(
    [test_base.reset_index(drop=True),
     X_test_rd.reset_index(drop=True)],
    axis=1
)

test_final.to_excel(
    "../../dataset/test set.xlsx",
    index=False
)

In [ ]:
nature_data = pd.read_excel("../../dataset/external validation/nature.xlsx")

In [ ]:
df_ex = nature_data.merge(
    plant_data,
    on="Species",
    how="left"
)
print(df_ex.shape)
#print(df_ex.head())

(2635, 798)


In [ ]:
df_ex.to_excel("../../dataset/external validation/nature_merged.xlsx", index=False)